<h2> Questions for Nick </h2>

- SAT and SBT - make summary table for nick, nick email JC
- SUP - are meds correct? - meds sent to Nick
- ECMO - where is the raw file? make clif table
- alcohol withdrawl, seizure, increased icp - icd10 code check

<h2> Import Libraries </h2>

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import math
from pandas.api.types import (
    is_numeric_dtype, is_bool_dtype, is_categorical_dtype,
    is_datetime64_any_dtype
)
import duckdb
import subprocess
import sys
import textwrap
import json

<h2> Global settings </h2>

In [ ]:
pd.set_option('display.max_columns', None)
con = duckdb.connect(database='ltvv.duckdb')
con.execute("SET TimeZone = 'America/Chicago'")

with open("config.json", "r", encoding="utf-8-sig") as f:
    cfg = json.load(f)

clif_path = cfg["paths"]["clif"]
raw_path = cfg["paths"]["raw"]
db_path = cfg["paths"]["db"]
print(clif_path)

<h2> Get initial vent cohort </h2>

In [ ]:
hourly_resp_path = f'{db_path}/clif_hourly_resp_support.parquet'
con.sql(f"SELECT * FROM '{hourly_resp_path}'")

<h3> Get all vent hospitalizations </h3>

In [ ]:
con.sql(f"""
-- Get all hosps with vent data
CREATE OR REPLACE VIEW vent_hosps AS
    SELECT DISTINCT(hospitalizations_joined_id)
    FROM '{hourly_resp_path}'
    WHERE device_category = 'imv';

CREATE OR REPLACE VIEW vent_data AS
    SELECT *
    FROM '{hourly_resp_path}'
    INNER JOIN vent_hosps USING (hospitalizations_joined_id);

SELECT * FROM vent_data;
 """).df()

<h3> Get hourly provider information </h3>

In [ ]:
provider_path = f'{clif_path}/clif_provider.parquet'
con.sql(f"SELECT * FROM '{provider_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW providers AS
    SELECT
        hospitalizations_joined_id,
        recorded_date,
        recorded_hour,
        prov_npi_shifted,
        prov_name_shifted,
        prov_spec_shifted
    FROM '{provider_path}';
SELECT * FROM providers;
 """)

<h3> Create initial hourly dataframe </h3>

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW initial_hourly_data AS
    SELECT *
    FROM vent_data
    LEFT JOIN providers USING (hospitalizations_joined_id, recorded_date, recorded_hour);
SELECT * FROM initial_hourly_data;
""")

In [ ]:
# USE THIS IF YOU WANT TO INCLUDE ALL ROWS WHERE TIDAL_VOLUME_SET ARE NOT NULL - USE BELOW CELL IF YOU WANT EITHER TIDAL_VOLUME_SET OR TIDAL_VOLUME_OBS!!
con.sql(f"""
CREATE OR REPLACE VIEW hourly_data AS
    SELECT *
    FROM initial_hourly_data
    WHERE vent_episode_id = '1'
        AND vent_episode_duration_hours >= '24'
        AND device_category = 'imv'
        AND tidal_volume_set IS NOT NULL
        AND location_category = 'icu'
    ORDER BY hospitalizations_joined_id, recorded_date, recorded_hour;

SELECT * FROM hourly_data;
""")

In [ ]:
# # USE THIS IF YOU WANT EITHER TIDAL_VOLUME_SET OR TIDAL_VOLUME_OBS!!
# con.sql(f"""
# CREATE OR REPLACE VIEW hourly_data AS
#     SELECT *
#     FROM initial_hourly_data
#     WHERE vent_episode_id = '1'
#         AND vent_episode_duration_hours >= '24'
#         AND location_category = 'icu'
#     ORDER BY hospitalizations_joined_id, recorded_date, recorded_hour;

# SELECT * FROM hourly_data;
# """)

In [ ]:
con.sql(f"""
-- STEP 1: Calculate the fifth vent hour per hospitalization
CREATE OR REPLACE VIEW fifth_hour AS
SELECT
    hospitalizations_joined_id,
    MIN(vent_episode_hour_seq) AS initial_vent_episode_hour_seq,
    MIN(vent_episode_hour_seq) + 4 AS four_hr_after_initial,
    MIN(vent_episode_hour_seq) + 4 AS vent_episode_hour_seq -- used for join
FROM hourly_data
GROUP BY hospitalizations_joined_id;

-- STEP 2: Inner join with main data to get physician_df
CREATE OR REPLACE VIEW physician_df AS
SELECT d.*
FROM hourly_data d
INNER JOIN fifth_hour f
  ON d.hospitalizations_joined_id = f.hospitalizations_joined_id
 AND d.vent_episode_hour_seq = f.vent_episode_hour_seq;

-- STEP 3: Get providers who started MV on 5th hour with ≥25 cases
CREATE OR REPLACE VIEW provider_counts AS
SELECT
    prov_npi_shifted,
    COUNT(*) AS count
FROM physician_df
GROUP BY prov_npi_shifted
HAVING count >= 25;

-- STEP 4: Keep only records within first 24 hours of vent AND among valid providers
CREATE OR REPLACE VIEW firstTime AS
SELECT p.*
FROM physician_df p
JOIN provider_counts pc
  ON p.prov_npi_shifted = pc.prov_npi_shifted
WHERE p.vent_episode_hospital_hour_seq <= 24;

-- STEP 5: Get all 2PM recorded_hour values
CREATE OR REPLACE VIEW data_2pm_all AS
SELECT *
FROM hourly_data
WHERE recorded_hour = 14.0;

-- STEP 6: Filter this 2PM set to only providers with ≥25 MV starts
CREATE OR REPLACE VIEW data_2pm_filtered AS
SELECT d.*
FROM data_2pm_all d
JOIN provider_counts pc
  ON d.prov_npi_shifted = pc.prov_npi_shifted;

-- STEP 7: Start with data_2pm_filtered, replace with firstTime records where there's a match
CREATE OR REPLACE VIEW initial_data AS
SELECT * FROM firstTime
UNION ALL
SELECT * FROM data_2pm_filtered d
WHERE NOT EXISTS (
    SELECT 1 FROM firstTime f 
    WHERE f.hospitalizations_joined_id = d.hospitalizations_joined_id 
    AND f.recorded_date = d.recorded_date
);
        
-- Step 8: Get final vent-prov dataframe        
CREATE OR REPLACE VIEW data AS
SELECT DISTINCT ON (hospitalizations_joined_id, date)
  hospitalizations_joined_id,
  recorded_date as date,
  recorded_date,
  recorded_hour as prov_assigned_hour,
  tidal_volume_set,
  (spo2 / fio2_set) AS sf_ratio,
  ibw,
  vent_episode_id,
  vent_episode_duration_hours,
  vent_episode_hour_seq,
  vent_episode_hospital_duration_hours,
  device_category as vent_device_category,
  mode_category as vent_mode_category,
  prov_npi_shifted,
  prov_name_shifted,
  prov_spec_shifted,
  location_category,
  hospital_id,
  tracheostomy
FROM initial_data;

SELECT * FROM data;
""")

In [ ]:
con.sql(f"SELECT COUNT(Distinct prov_npi_shifted) from data")

<h2> Patient data </h2>

In [ ]:
patient_path = f'{clif_path}/clif_patient.parquet'
con.sql(f"SELECT * FROM '{patient_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW patient AS
    SELECT DISTINCT ON (mdm_link_id)
        mdm_link_id,
        sex_category,
        language_name,
        race_category,
        ethnicity_category,
        death_date,
        birth_date
    FROM '{patient_path}';
SELECT * FROM patient;
""")

<h2> Proning </h2>

In [ ]:
position_path = f'{clif_path}/clif_position.parquet'
con.sql(f"SELECT * FROM '{position_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW proning AS
    SELECT DISTINCT ON (hospitalizations_joined_id, date)
        hospitalizations_joined_id,
        date(recorded_dttm) as date,
        TRUE as prone_position
    FROM '{position_path}'
    WHERE position_category = 'prone';
SELECT * FROM proning;
""")

<h2> Hospitalization data </h2>

In [ ]:
hosp_path = f'{clif_path}/clif_hospitalization.parquet'
con.sql(f"SELECT * FROM '{hosp_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW elix_vw AS
    SELECT DISTINCT ON (hospitalizations_joined_id)
        hospitalizations_joined_id,
        patient_id as mdm_link_id,
        admission_dttm,
        discharge_dttm,
        DATE_DIFF('min', admission_dttm, discharge_dttm) as hosp_los_min,
        YEAR(admission_dttm) as admit_year,
        age_at_admission,
        charlson,
        elix_vw
    FROM '{hosp_path}'
    ORDER BY hospitalizations_joined_id, admission_dttm DESC;
SELECT * FROM elix_vw;
""")

<h2> Save LAPS2 hospitalizations </h2>

In [ ]:
con.sql(f"""
WITH base AS (
    SELECT distinct hospitalizations_joined_id
    FROM data
),
hospitalizations AS (
    SELECT *
    FROM '{hosp_path}'
)
SELECT
    base.hospitalizations_joined_id,
    MODE(hospitalizations.patient_id) as patient_id,
    MODE(hospitalizations.patient_id) as mdm_link_id
FROM base
LEFT JOIN hospitalizations USING (hospitalizations_joined_id)
GROUP BY hospitalizations_joined_id
""").df().to_parquet('laps2_hospitalizations.parquet')

<h2> Vitals </h2>

In [ ]:
vitals_path = f"{clif_path}/clif_vitals.parquet"
con.sql(f"SELECT * FROM '{vitals_path}' LIMIT 1000")

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW vitals AS
    SELECT
        hospitalizations_joined_id,
        DATE(recorded_dttm) AS date,

        -- height
        AVG(CASE WHEN vital_category = 'height_cm' THEN vital_value ELSE NULL END) AS height_cm,

        -- weight
        AVG(CASE WHEN vital_category = 'weight_kg' THEN vital_value ELSE NULL END) AS weight_kg,

        -- respiratory rate
        MAX(CASE WHEN vital_category = 'respiratory_rate' THEN vital_value ELSE NULL END) AS respiratory_rate_max,
        MIN(CASE WHEN vital_category = 'respiratory_rate' THEN vital_value ELSE NULL END) AS respiratory_rate_min,
        AVG(CASE WHEN vital_category = 'respiratory_rate' THEN vital_value ELSE NULL END) AS respiratory_rate_avg,
                
        -- spo2
        MAX(CASE WHEN vital_category = 'spo2' THEN vital_value ELSE NULL END) AS spo2_max,
        MIN(CASE WHEN vital_category = 'spo2' THEN vital_value ELSE NULL END) AS spo2_min,
        AVG(CASE WHEN vital_category = 'spo2' THEN vital_value ELSE NULL END) AS spo2_avg

    FROM '{vitals_path}'
    GROUP BY hospitalizations_joined_id, DATE(recorded_dttm);
""")
con.sql('SELECT * FROM vitals')

<h2> Labs </h2>

In [ ]:
labs_path = f'{clif_path}/clif_labs.parquet'
duckdb.sql(f"SELECT * FROM '{labs_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW labs AS (
    SELECT 
        hospitalizations_joined_id,
        DATE(lab_collect_dttm) AS date,
               
        -- ph_arterial
        MIN(CASE WHEN lab_category = 'ph_arterial' THEN lab_value_numeric END) AS ph_arterial_min,
        AVG(CASE WHEN lab_category = 'ph_arterial' THEN lab_value_numeric END) AS ph_arterial_avg,
        MAX(CASE WHEN lab_category = 'ph_arterial' THEN lab_value_numeric END) AS ph_arterial_max,
        MAX(CASE WHEN lab_category = 'ph_arterial' THEN reference_unit END) AS ph_arterial_unit,

        -- so2_arterial
        MIN(CASE WHEN lab_category = 'so2_arterial' THEN lab_value_numeric END) AS so2_arterial_min,
        AVG(CASE WHEN lab_category = 'so2_arterial' THEN lab_value_numeric END) AS so2_arterial_avg,
        MAX(CASE WHEN lab_category = 'so2_arterial' THEN lab_value_numeric END) AS so2_arterial_max,
        MAX(CASE WHEN lab_category = 'so2_arterial' THEN reference_unit END) AS so2_arterial_unit,

        -- pco2_arterial
        MIN(CASE WHEN lab_category = 'pco2_arterial' THEN lab_value_numeric END) AS pco2_arterial_min,
        AVG(CASE WHEN lab_category = 'pco2_arterial' THEN lab_value_numeric END) AS pco2_arterial_avg,
        MAX(CASE WHEN lab_category = 'pco2_arterial' THEN lab_value_numeric END) AS pco2_arterial_max,
        MAX(CASE WHEN lab_category = 'pco2_arterial' THEN reference_unit END) AS pco2_arterial_unit,

        -- po2_arterial
        MIN(CASE WHEN lab_category = 'po2_arterial' THEN lab_value_numeric END) AS po2_arterial_min,
        AVG(CASE WHEN lab_category = 'po2_arterial' THEN lab_value_numeric END) AS po2_arterial_avg,
        MAX(CASE WHEN lab_category = 'po2_arterial' THEN lab_value_numeric END) AS po2_arterial_max,
        MAX(CASE WHEN lab_category = 'po2_arterial' THEN reference_unit END) AS po2_arterial_unit

    FROM '{labs_path}'
    WHERE lab_category ILIKE '%arterial%'
    GROUP BY hospitalizations_joined_id, date
    );
SELECT * FROM labs;
""")


<h2> SOFA </h2>

In [ ]:
sofa_path = f'{raw_path}/acute_sofascore.parquet'
duckdb.sql(f"SELECT * FROM '{sofa_path}' LIMIT 1000")

In [ ]:
# Get min, max, and avg sofa scores by day
con.sql(f"""
    CREATE OR REPLACE VIEW sofa AS (
        SELECT 
            hospitalizations_joined_id,
            DATE(recorded_time) AS date,
            MIN(sofa_score) AS sofa_score_min,
            MAX(sofa_score) AS sofa_score_max,
            AVG(sofa_score) AS sofa_score_mean,
        FROM '{sofa_path}'
        GROUP BY hospitalizations_joined_id, date
    )
""")
con.sql('SELECT * FROM sofa')

<h2> ECMO </h2>

In [ ]:
ecmo is wrong.......
ecmo_path = f"{raw_path}/acute_ventdata.parquet"

con.sql(f"""
CREATE OR REPLACE VIEW ecmo AS
    SELECT DISTINCT ON (mdm_link_id, date)
        mdm_link_id,
        date(recorded_time) as date,
        TRUE as ecmo
    FROM '{ecmo_path}'
    WHERE flo_meas_name ILIKE '%ecmo%';
SELECT * FROM ecmo;
""")

<h2> Meds </h2>

<h3> Sup </h3>

In [ ]:
meds_continuous_path = f'{clif_path}/clif_medication_admin_continuous.parquet'
con.sql(f"SELECT * FROM '{meds_continuous_path}'")

In [ ]:
con.sql(f"""
    SELECT *
    FROM '{meds_continuous_path}'
    WHERE med_category IN (
        'omeprazole',
        'pantoprazole',
        'esomeprazole',
        'lansoprazole',
        'rabeprazole',
        'famotidine',
        'ranitidine',
        'nizatidine',
        'cimetidine'
    );
""").df()['med_category'].value_counts()

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW continuous_sup AS
    SELECT DISTINCT ON (hospitalizations_joined_id, date)
        hospitalizations_joined_id,
        date(admin_dttm) as date,
        TRUE as ppi_or_h2_continuous
    FROM '{meds_continuous_path}'
    WHERE med_category IN (
        'omeprazole',
        'pantoprazole',
        'esomeprazole',
        'lansoprazole',
        'rabeprazole',
        'famotidine',
        'ranitidine',
        'nizatidine',
        'cimetidine'
    );
SELECT * FROM continuous_sup;
""")

In [ ]:
meds_intermittent_path = f'{clif_path}/clif_medication_admin_intermittent.parquet'
con.sql(f"SELECT * FROM '{meds_intermittent_path}'")

In [ ]:
con.sql(f"""
    SELECT *
    FROM '{meds_intermittent_path}'
    WHERE med_category IN (
        'omeprazole',
        'pantoprazole',
        'esomeprazole',
        'lansoprazole',
        'rabeprazole',
        'famotidine',
        'ranitidine',
        'nizatidine',
        'cimetidine'
    );
""").df()['med_category'].value_counts()

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW intermittent_sup AS
    SELECT DISTINCT ON (hospitalizations_joined_id, date)
        hospitalizations_joined_id,
        date(admin_dttm) as date,
        TRUE as ppi_or_h2_intermittent
    FROM '{meds_intermittent_path}'
    WHERE med_category IN (
        'omeprazole',
        'pantoprazole',
        'esomeprazole',
        'lansoprazole',
        'rabeprazole',
        'famotidine',
        'ranitidine',
        'nizatidine',
        'cimetidine'
    );
SELECT * FROM intermittent_sup
""")

<h3> Paralytics </h3>

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW paralytics_continuous AS
    SELECT
        hospitalizations_joined_id,
        DATE(admin_dttm) as date,
        TRUE as paralytics_continuous
    FROM '{meds_continuous_path}'
    WHERE med_group ILIKE 'paralytics'
        AND med_dose != '0'
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM paralytics_continuous;
""")

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW paralytics_intermittent AS
    SELECT
        hospitalizations_joined_id,
        DATE(admin_dttm) as date,
        TRUE as paralytics_intermittent
    FROM '{meds_intermittent_path}'
    WHERE med_group ILIKE 'paralytics'
        AND med_dose != '0'
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM paralytics_intermittent;
""")

<h3> Sedation </h3>

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW continuous_sedation AS
    SELECT DISTINCT ON (hospitalizations_joined_id, date)
        hospitalizations_joined_id,
        date(admin_dttm) as date,
        TRUE as sedation_continuous_meds
    FROM '{meds_continuous_path}'
    WHERE med_group = 'sedation'
        AND med_dose != '0';
SELECT * FROM continuous_sedation;
""")

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW intermittent_sedation AS
    SELECT DISTINCT ON (hospitalizations_joined_id, date)
        hospitalizations_joined_id,
        date(admin_dttm) as date,
        TRUE as sedation_intermittent_meds
    FROM '{meds_intermittent_path}'
    WHERE med_group in ('sedation', 'analgesia')
        AND med_dose != '0';
SELECT * FROM intermittent_sedation;
""")

<h2> ICD10 codes </h2>

In [ ]:
diag_path = f"{clif_path}/clif_admission_diagnosis.parquet"
con.sql(f"SELECT * FROM '{diag_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW diagnosis_codes AS
WITH row_flags AS (
  SELECT
    hospitalizations_joined_id,
    -- Alcohol withdrawal (row-level)
    CASE WHEN
         diagnosis_code ILIKE 'F10.130%' OR diagnosis_code ILIKE 'F10.131%' OR
         diagnosis_code ILIKE 'F10.132%' OR diagnosis_code ILIKE 'F10.139%' OR
         diagnosis_code ILIKE 'F10.230%' OR diagnosis_code ILIKE 'F10.231%' OR
         diagnosis_code ILIKE 'F10.232%' OR diagnosis_code ILIKE 'F10.239%' OR
         diagnosis_code ILIKE 'F10.930%' OR diagnosis_code ILIKE 'F10.931%' OR
         diagnosis_code ILIKE 'F10.932%'
    THEN 1 ELSE 0 END AS alcohol_withdrawal_flag,

    -- Seizure (row-level)
    CASE WHEN
         diagnosis_code ILIKE 'R56.9%' OR diagnosis_code ILIKE 'G40.909%'
    THEN 1 ELSE 0 END AS seizure_flag,

    -- Increased ICP-related (row-level)
    CASE WHEN
         diagnosis_code ILIKE 'G93.2%' OR diagnosis_code ILIKE 'G93.6%' OR
         diagnosis_code ILIKE 'G93.5%' OR diagnosis_code ILIKE 'H47.11%'
    THEN 1 ELSE 0 END AS icp_flag
  FROM '{diag_path}'
)
SELECT
  hospitalizations_joined_id,
  CASE WHEN MAX(alcohol_withdrawal_flag) = 1 THEN 1 ELSE 0 END AS alcohol_withdrawal_flag,
  CASE WHEN MAX(seizure_flag)            = 1 THEN 1 ELSE 0 END AS seizure_flag,
  CASE WHEN MAX(icp_flag)                = 1 THEN 1 ELSE 0 END AS icp_flag
FROM row_flags
GROUP BY hospitalizations_joined_id
;

-- View results (order at query time)
SELECT * FROM diagnosis_codes;
""")


<h2> Glucose control </h2>

In [ ]:
# Look for range between 80 - 180. Get min, max, mean, and median for day. Ever fall below 80 that day? Ever go above 180 that day?

In [ ]:
labs_path = f"{clif_path}/clif_labs.parquet"
con.sql(f"SELECT * FROM '{labs_path}'")

In [ ]:
con.sql(f"""
    SELECT *
    FROM '{labs_path}'
    WHERE lab_category ILIKE '%glucose%'
""").df()['lab_category'].value_counts()

In [ ]:
con.sql(f"SELECT * FROM '{labs_path}' WHERE lab_category ILIKE '%glucose%'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW glucose AS
    SELECT
        hospitalizations_joined_id,
        date(lab_collect_dttm) as date,
        MIN(lab_value_numeric) as glucose_daily_min,
        MEAN(lab_value_numeric) as glucose_daily_mean,
        MEDIAN(lab_value_numeric) as glucose_daily_median,
        MAX(lab_value_numeric) as glucose_daily_max,
        MAX(reference_unit) as glucose_units
    FROM '{labs_path}'
    WHERE lab_category ILIKE '%glucose%'
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM glucose;
""")

<h2> SAT SBT </h2>

In [ ]:
assessment_path = f'{clif_path}/clif_patient_assessments.parquet'
con.sql(f"SELECT * FROM '{assessment_path}'")

In [ ]:
con.sql(f"SELECT * FROM '{assessment_path}' WHERE assessment_category = 'RASS'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW rass AS
    SELECT
        hospitalizations_joined_id,
        date(recorded_dttm) as date,
        MIN(numerical_value) as daily_rass_min,
        MAX(numerical_value) as daily_rass_max
    FROM '{assessment_path}'
    WHERE assessment_category = 'RASS'
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM rass;
""")

In [ ]:
con.sql(f"""
    SELECT
        assessment_name,
        assessment_category,
        assessment_group,
        MAX(flo_meas_id) as flo_meas_id,
        MIN(numerical_value) as min_numerical_value,
        MEDIAN(numerical_value) as median_numerical_value,
        MAX(numerical_value) as max_numerical_value,
        MEAN(numerical_value) as mean_numerical_value,
        list(distinct text_value) as text_value_examples,
        list(distinct categorical_value) as categorical_value_examples,
        COUNT(*) as counts
    FROM '{assessment_path}'
    GROUP BY assessment_name, assessment_category, assessment_group
    ORDER BY assessment_group
""").df()#.to_csv('patient_assessment_summary.csv')

In [ ]:
con.sql(f"""
    SELECT assessment_category
    FROM '{assessment_path}'
    WHERE assessment_category ILIKE '%sbt%'
""").df().value_counts()
# sbt delivery: assessment_group, categorical_value: yes
#

In [ ]:
con.sql(f"""
    SELECT distinct assessment_group
    FROM '{assessment_path}'
    WHERE assessment_group ILIKE '%sbt%'
""")
# sbt delivery: assessment_group, categorical_value: yes
#

In [ ]:
con.sql(f"""
    SELECT distinct assessment_category
    FROM '{assessment_path}'
    WHERE assessment_category ILIKE '%sbt%'
""")
# sbt delivery: assessment_group, categorical_value: yes
#

In [ ]:
con.sql(f"""
    SELECT distinct assessment_category
    FROM '{assessment_path}'
    WHERE assessment_category ILIKE '%sbt%'
""")
# sbt delivery: assessment_group, categorical_value: yes
#

In [ ]:
con.sql(f"""
    SELECT *
    FROM '{assessment_path}'
    WHERE assessment_group ILIKE '%sat%'
""")

In [ ]:
con.sql(f"""
    SELECT *
    FROM '{assessment_path}'
    WHERE assessment_category ILIKE '%rass%'
""")

<h2> Full Vent data </h2>

In [ ]:
duckdb.sql(f"SELECT * FROM '{hourly_resp_path}'LIMIT 1000;")

In [ ]:
# Get vent info
con.sql(f"""
CREATE OR REPLACE VIEW all_vent_data AS
    SELECT 
        hospitalizations_joined_id,
        recorded_date AS date,
        COUNT(recorded_hour) AS daily_hours_on_vent,
        MAX(vent_episode_id) as daily_last_vent_episode_id,
        MODE(mode_category) as primary_vent_mode_category,
        MIN(fio2_set) as vent_fio2_min,
        MAX(fio2_set) as vent_fio2_max,
        MEAN(fio2_set) as vent_fio2_mean,
        MIN(peep_set) AS vent_peep_min,
        MAX(peep_set) AS vent_peep_max,
        MEAN(peep_set) AS vent_peep_mean,
        MIN(tidal_volume_set) AS vent_tidal_volume_min,
        MAX(tidal_volume_set) AS vent_tidal_volume_max,
        MEAN(tidal_volume_set) AS vent_tidal_volume_mean,

        -- sf_ratio aggregations
        MIN(spo2 / fio2_set) AS sf_ratio_daily_min,
        MAX(spo2 / fio2_set) AS sf_ratio_daily_max,
        AVG(spo2 / fio2_set) AS sf_ratio_daily_mean
    FROM '{hourly_resp_path}'
    WHERE device_category = 'imv'
    GROUP BY hospitalizations_joined_id, date;

SELECT * FROM all_vent_data;
""")

<h2> Get mode hours </h2>

In [ ]:
con.sql(f"SELECT distinct mode_category From '{hourly_resp_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW imv_hours AS
SELECT
    hospitalizations_joined_id,
    COUNT(*) AS imv_hours,

    SUM(CASE WHEN mode_category = 'nava'                              THEN 1 ELSE 0 END) AS nava_hours,
    SUM(CASE WHEN mode_category = 'blow by'                           THEN 1 ELSE 0 END) AS blow_by_hours,
    SUM(CASE WHEN mode_category = 'simv'                              THEN 1 ELSE 0 END) AS simv_hours,
    SUM(CASE WHEN mode_category = 'other'                             THEN 1 ELSE 0 END) AS other_hours,
    SUM(CASE WHEN mode_category = 'assist control-volume control'     THEN 1 ELSE 0 END) AS acvc_hours,
    SUM(CASE WHEN mode_category = 'pressure-regulated volume control' THEN 1 ELSE 0 END) AS prvc_hours,
    SUM(CASE WHEN mode_category = 'aprv'                              THEN 1 ELSE 0 END) AS aprv_hours,
    SUM(CASE WHEN mode_category = 'pressure control'                  THEN 1 ELSE 0 END) AS pressure_control_hours,
    SUM(CASE WHEN mode_category = 'pressure support/cpap'             THEN 1 ELSE 0 END) AS pressure_support_cpap_hours,

    -- optional: count rows where mode_category is NULL
    SUM(CASE WHEN mode_category IS NULL THEN 1 ELSE 0 END) AS mode_missing_hours

FROM '{hourly_resp_path}'
WHERE device_category = 'imv'
    AND vent_episode_id = '1'
GROUP BY hospitalizations_joined_id;
SELECT * FROM imv_hours;
""")


<h2> Vent Mode </h2>

In [ ]:
resp_supp_path = f"{clif_path}/clif_respiratory_support.parquet"
con.sql(f"SELECT * FROM '{resp_supp_path}'")

In [ ]:
con.sql(f"SELECT distinct mode_category FROM '{resp_supp_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW vent_mode AS
    SELECT
        hospitalizations_joined_id,
        DATE(recorded_dttm) as date,
        MODE(mode_category) primary_mode_category
    FROM '{resp_supp_path}'
    WHERE device_category ILIKE '%imv%'
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM vent_mode;
""")

<h3> Get observed tidal volumes </h3>

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW tidal_volume_obs AS
    SELECT
        hospitalizations_joined_id,
        recorded_dttm as tidal_obs_dttm,
        tidal_volume_obs
    FROM '{resp_supp_path}'
    WHERE tidal_volume_obs IS NOT NULL;
SELECT * FROM tidal_volume_obs;
""")

<h2> ICU </h2>

In [ ]:
adt_path = clif_path + '/clif_adt.parquet'
con.sql(f"SELECT * FROM '{adt_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW ever_in_icu_hosp AS
    SELECT DISTINCT ON (hospitalizations_joined_id)
        hospitalizations_joined_id,
        TRUE as ever_in_icu_hosp
    FROM '{adt_path}'
    WHERE location_category = 'icu';
SELECT * FROM ever_in_icu_hosp;
""")

<h2> LAPS2 </h2>

<h3> Call laps2_wrangler_ltvv to create laps2_data_ltvv.parquet </h3>

In [ ]:
!run_laps2.bat

<h3> Laps2 view creation </h3>

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW laps2 AS
    SELECT DISTINCT ON (hospitalizations_joined_id, recorded_date)
        hospitalizations_joined_id,
        recorded_date as date,
        recorded_date,
        laps2
    FROM 'laps2_data.parquet';
SELECT * FROM laps2;
""")

<h2> Covid - using non-culture micro table from different refresh so hospitalizations_joined_id WILL NOT BE USABLE!! </h2>

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW covid AS
    SELECT DISTINCT ON (mdm_link_id, date)
        patient_id as mdm_link_id,
        DATE(collect_dttm) as date,
        DATE(collect_dttm) as covid_date,
        TRUE as covid
    FROM 'Z:/DataStageData/CQODE DB Backbone/rclif/clif_microbiology_nonculture.parquet'
    WHERE micro_category ILIKE '%sars_cov%'
        AND result_category = 'Detected';
SELECT * FROM covid;
""")

<h2> Merge tables and form final dataframe </h2>

In [ ]:
print(con.sql("SELECT COUNT(*) FROM data"))

<h3> Create static vs longitudinal ids </h3>

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW static_ids AS
    SELECT DISTINCT hospitalizations_joined_id
    FROM data;
        
CREATE OR REPLACE VIEW longitudinal_ids AS
    SELECT DISTINCT ON (hospitalizations_joined_id, date)
        hospitalizations_joined_id,
        date
    FROM data;
    
SELECT * FROM longitudinal_ids;
""")

<h3> Creating final table </h3>

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW static AS
    SELECT
        static_ids.hospitalizations_joined_id,
        elix_vw.* EXCLUDE (hospitalizations_joined_id),
        diagnosis_codes.* EXCLUDE (hospitalizations_joined_id),
        ever_in_icu_hosp.* EXCLUDE (hospitalizations_joined_id),
        imv_hours.* EXCLUDE(hospitalizations_joined_id)
    FROM static_ids
        LEFT JOIN elix_vw USING (hospitalizations_joined_id)
        LEFT JOIN diagnosis_codes USING (hospitalizations_joined_id)
        LEFT JOIN ever_in_icu_hosp USING (hospitalizations_joined_id)
        LEFT JOIN imv_hours USING (hospitalizations_joined_id);

CREATE OR REPLACE VIEW longitudinal AS
    SELECT
        longitudinal_ids.hospitalizations_joined_id,
        longitudinal_ids.date,
        all_vent_data.* EXCLUDE (hospitalizations_joined_id, date),
        vent_mode.* EXCLUDE (hospitalizations_joined_id, date),
        vitals.* EXCLUDE (hospitalizations_joined_id, date, height_cm, weight_kg),
        labs.* EXCLUDE (hospitalizations_joined_id, date),
        proning.* EXCLUDE (hospitalizations_joined_id, date),
        continuous_sup.* EXCLUDE (hospitalizations_joined_id, date),
        intermittent_sup.* EXCLUDE(hospitalizations_joined_id, date),
        paralytics_continuous.* EXCLUDE (hospitalizations_joined_id, date),
        paralytics_intermittent.* EXCLUDE (hospitalizations_joined_id, date),
        glucose.* EXCLUDE (hospitalizations_joined_id, date),
        rass.* EXCLUDE (hospitalizations_joined_id, date),
        continuous_sedation.* EXCLUDE (hospitalizations_joined_id, date),
        intermittent_sedation.* EXCLUDE (hospitalizations_joined_id, date),
        laps2.* EXCLUDE (hospitalizations_joined_id, date)
    FROM longitudinal_ids
        LEFT JOIN all_vent_data USING (hospitalizations_joined_id, date)
        LEFT JOIN vent_mode USING (hospitalizations_joined_id, date)
        LEFT JOIN vitals USING (hospitalizations_joined_id, date)
        LEFT JOIN labs USING (hospitalizations_joined_id, date)
        LEFT JOIN proning USING (hospitalizations_joined_id, date)
        LEFT JOIN continuous_sup USING (hospitalizations_joined_id, date)
        LEFT JOIN intermittent_sup USING (hospitalizations_joined_id, date)
        LEFT JOIN paralytics_continuous USING (hospitalizations_joined_id, date)
        LEFT JOIN paralytics_intermittent USING (hospitalizations_joined_id, date)
        LEFT JOIN glucose USING (hospitalizations_joined_id, date)
        LEFT JOIN rass USING (hospitalizations_joined_id, date)
        LEFT JOIN continuous_sedation USING (hospitalizations_joined_id, date)
        LEFT JOIN intermittent_sedation USING (hospitalizations_joined_id, date)
        LEFT JOIN laps2 USING (hospitalizations_joined_id, date);
    
CREATE OR REPLACE VIEW final_df AS
    SELECT
        data.*,
        static.* EXCLUDE (hospitalizations_joined_id),
        longitudinal.* EXCLUDE (hospitalizations_joined_id, date),
        patient.* EXCLUDE (mdm_link_id),
        ecmo.* EXCLUDE (mdm_link_id, date)
    FROM data
        LEFT JOIN static USING (hospitalizations_joined_id)
        LEFT JOIN longitudinal USING (hospitalizations_joined_id, date)
        LEFT JOIN patient USING (mdm_link_id)
        LEFT JOIN ecmo USING (mdm_link_id, date);
""")

In [ ]:
# print(con.sql(f"SELECT COUNT(*) FROM static_ids"))
# print(con.sql(f"SELECT COUNT(*) FROM static"))

# print(con.sql(f"SELECT COUNT(*) FROM longitudinal_ids"))
# print(con.sql(f"SELECT COUNT(*) FROM longitudinal"))

# print(con.sql(f"SELECT COUNT(*) FROM final_df"))

In [ ]:
con.sql(f"""
CREATE OR REPLACE TABLE final_table AS
    SELECT *
    FROM final_df;
SELECT * FROM final_table;
""")

In [ ]:
print(con.sql(f"SELECT COUNT(*) FROM data"))
print(con.sql(f"SELECT COUNT(*) FROM final_table"))

In [ ]:
data = con.sql(f"SELECT * FROM final_table;").df().sort_values(by=['hospitalizations_joined_id', 'date'])
data

<h3> Merge in NEAREST height and weight </h3>

In [ ]:
# Get all height and weight values
df = con.sql("""
             SELECT *
             FROM vitals
             ORDER BY date
""").df()[['hospitalizations_joined_id', 'date', 'height_cm', 'weight_kg']].sort_values(by='date').reset_index(drop=True)

# Get all non-na height and weight values
height_df = df[~df['height_cm'].isna()][['hospitalizations_joined_id', 'date', 'height_cm']]
weight_df = df[~df['weight_kg'].isna()][['hospitalizations_joined_id', 'date', 'weight_kg']]

# Merge nearest non-na heights and weights
data = data.sort_values(by='date').reset_index(drop=True)

data = pd.merge_asof(
    data,
    height_df,
    by='hospitalizations_joined_id',
    on='date',
    direction='nearest',
    allow_exact_matches=True
)

data = pd.merge_asof(
    data,
    weight_df,
    by='hospitalizations_joined_id',
    on='date',
    direction='nearest',
    allow_exact_matches=True
)

data = data.sort_values(by=['hospitalizations_joined_id', 'date'])
data.head(5)

<h3> Merge in sofa </h3>

In [ ]:
df = con.sql("""
             SELECT *
             FROM sofa
             ORDER BY date
""").df()
data = data.sort_values(by='date')

# Merge within 2 days before or after if missing
data = pd.merge_asof(
    data,
    df,
    by='hospitalizations_joined_id',
    on='date',
    direction='nearest',
    tolerance=pd.Timedelta('2D')
)
data = data.sort_values(by=['hospitalizations_joined_id', 'date'])
data.head(5)

<h2> Merging in nearest prior observed tidal volume </h2>

In [ ]:
# Get tidal_volume_obs and order by recorded_dttm
df = con.sql("""
             SELECT *
             FROM tidal_volume_obs
             ORDER BY tidal_obs_dttm
""").df()

# Make a datetime from the date and hour, sort dttm
data['tidal_obs_dttm'] = (
    pd.to_datetime(data['recorded_date']) + 
    pd.to_timedelta(data['prov_assigned_hour'].astype('int64'), unit='h')
).dt.tz_localize('America/Chicago', ambiguous=False, nonexistent='shift_forward')

# Precision cast
data['tidal_obs_dttm'] = data['tidal_obs_dttm'].astype('datetime64[us, America/Chicago]')
df['tidal_obs_dttm'] = df['tidal_obs_dttm'].astype('datetime64[us, America/Chicago]')
data = data.sort_values(by='tidal_obs_dttm')

# Merge within 3 hours before 
data = pd.merge_asof(
    data,
    df,
    by='hospitalizations_joined_id',
    on='tidal_obs_dttm',
    direction='backward',
    tolerance=pd.Timedelta('3h'),
    allow_exact_matches=True
)
data = data.sort_values(by=['hospitalizations_joined_id', 'date'])
data[['hospitalizations_joined_id', 'date', 'tidal_obs_dttm', 'tidal_volume_obs', 'tidal_volume_set']].head(5)

In [ ]:
# when is tidal_obs and not tidal_vol set > that is the non-volume control number

In [ ]:
data[['hospitalizations_joined_id', 'date', 'tidal_obs_dttm', 'tidal_volume_obs', 'tidal_volume_set']]

<h3> Merge in covid data within 2 weeks from first patient day. Carry forward? </h3>

In [ ]:
df = con.sql("""
             SELECT *
             FROM covid
             ORDER BY date
""").df().dropna()

intubation = data.sort_values(by='date').drop_duplicates(subset='hospitalizations_joined_id', keep='first')

# Merge within 14 days of intubation
intubation = pd.merge_asof(
    intubation,
    df,
    by='mdm_link_id',
    on='date',
    direction='backward',
    tolerance=pd.Timedelta('14D'),
    allow_exact_matches=True
)
intubation = intubation.sort_values(by=['hospitalizations_joined_id', 'date'])
print(f'Hospitalizations positive for covid: {len(intubation[intubation['covid'] == True])}')

intubation[intubation['covid'] == True][['hospitalizations_joined_id', 'mdm_link_id', 'date', 'covid_date', 'covid']]

In [ ]:
# Merge back onto main dataframe
data = data.merge(intubation[['hospitalizations_joined_id', 'covid']], on='hospitalizations_joined_id', how='left')
data[data['covid'] == True]

<h2> Pandas engineering </h2>

<h3> Base engineering </h3>

In [ ]:
# Get BMI
data['bmi'] = round(data['weight_kg'] / (data['height_cm'] / 100) ** 2, 2)

# PF ratio
data['pf_ratio_max'] = data['po2_arterial_max'] / data['vent_fio2_min']
data['pf_ratio_min'] = data['po2_arterial_min'] / data['vent_fio2_max']

# PPI or H2
data['ppi_or_h2'] = data['ppi_or_h2_continuous'] | data['ppi_or_h2_intermittent']

# Paralytics
data['paralytics'] = data['paralytics_continuous'] | data['paralytics_intermittent']

# Sedation

# ECMO
data['ecmo'] = data['ecmo'].fillna(False)

# COVID - fill in False
data['covid'] = data['covid'].fillna(False)

# Calculate in hospital mortality
# data['death_date'] = pd.to_datetime(data['death_date'])
# data['mortality_inhospital'] = data['death_date'] <= data['discharge_dttm']
data['mortality_inhospital'] = data['death_date'].dt.date <= data['discharge_dttm'].dt.date

# Get hospital length of stay in days
data['hosp_los_days'] = data['hosp_los_min'] / 1440

# Tidal volume set or tidal volume observed
data['tidal_volume_set_or_obs'] = data['tidal_volume_set'].fillna(data['tidal_volume_obs'])

# Engineer ltvv with tidal_volume_set filled with tidal_volume_obs
data['tidal_volume_set_or_obs_ibw'] = data['tidal_volume_set_or_obs'] / data['ibw']
data['ltvv_tidal_volume_set_or_obs_8'] = (data['tidal_volume_set_or_obs_ibw'] <= 8).astype(int)
data['ltvv_tidal_volume_set_or_obs_6'] = (data['tidal_volume_set_or_obs_ibw'] <= 6).astype(int)


# Engineer ltvv with tidal_volume_set only
data['tidal_volume_set_ibw'] = data['tidal_volume_set'] / data['ibw']
data['ltvv_8'] = (data['tidal_volume_set_ibw'] <= 8).astype(int)
data['ltvv_6'] = (data['tidal_volume_set_ibw'] <= 6).astype(int)

# Save
# data.to_parquet('vent_ebp_data.parquet')

data.head(5)

<h3> Carry forward sf_ratio min, pf ratio min, ph min, and pco2 max up to 3 days </h3>

In [ ]:
# Carry forward ph and pco2 up to 3 days max
data = data.sort_values(by=['hospitalizations_joined_id', 'date'])

data['ph_arterial_min'] = (
    data.groupby('hospitalizations_joined_id')['ph_arterial_min']
    .ffill(limit=3)
)
data['pco2_arterial_max'] = (
    data.groupby('hospitalizations_joined_id')['pco2_arterial_max']
    .ffill(limit=3)
)

# Carry forward sf_ratio and pf_ratio_daily_min up to 3 days max
data['sf_ratio'] = (
    data.groupby('hospitalizations_joined_id')['sf_ratio']
    .ffill(limit=3)
)
data['pf_ratio_min'] = (
    data.groupby('hospitalizations_joined_id')['pf_ratio_min']
    .ffill(limit=3)
)

data

<h2> Assign rows as initial or subsequent </h2>

In [ ]:
day1_row = data.sort_values(by=['hospitalizations_joined_id', 'date']).drop_duplicates(subset='hospitalizations_joined_id') # get all first rows
day1_row = day1_row[day1_row['vent_episode_hour_seq'] <= 24] # keep only initial rows within 1 day of being on vent
day1_row['initial_vent_day'] = True
day1_row

In [ ]:
data = data.merge(day1_row[['hospitalizations_joined_id', 'date', 'initial_vent_day']], on=['hospitalizations_joined_id', 'date'], how='left')
data['initial_vent_day'] = data['initial_vent_day'].fillna(False)
data[['hospitalizations_joined_id', 'date', 'vent_episode_hour_seq', 'initial_vent_day']]

<h3> Map hospital ids </h3>

In [ ]:
hosp_ids = {
    'Woodwinds' : 'Hospital 1',
    'St. Johns' : 'Hospital 2',
    'Southdale' : 'Hospital 3',
    'Riverside' : 'Hospital 4',
    'Ridges' : 'Hospital 5',
    'Ranges/Hibbing' : 'Hospital 6',
    'Northland' : 'Hospital 7',
    'Lakes' : 'Hospital 8',
    'Grand Itasca' : 'Hospital 9',
    'UMMC' : 'Hospital Ref',
    'St. Joes' : 'delete',
    'Bethesda' : 'delete'
}
data['hospital_id'] = data['hospital_id'].map(hosp_ids)
data[['hospital_id']]

<h2> Final exclusions </h2>

In [ ]:
# DO exclusion > make sure all exclusions are done > 

In [ ]:
# Include only if tidal_volume_set_or_obs are not null
# data = data[~data['tidal_volume_set_or_obs'].isna()]
# data

In [ ]:
# Make sure 18+
data = data[data['age_at_admission'] >= 18]

# Make sure not tracheostomy
data = data[data['tracheostomy'] != 1]

# Remove ECMO patient days
data = data[data['ecmo'] != True]

# Remove st. joes and bethesda
data = data[data['hospital_id'] != 'delete']

# Remove hospital 9 for subsequent HRF-6 and HRF-8 model convergence
data = data[data['hospital_id'] != 'Hospital 9']

# Remove weird stuff
data = data.dropna(subset=['prov_npi_shifted'])
data = data[~data['prov_npi_shifted'].isin(['None', 'none', 'nan'])]

data

<h2> Edit column names </h2>

In [ ]:
# Get counts where set_tidal is null but tidal_obs is not!

In [ ]:
data

In [ ]:
# Use the same column names as before
data['age'] = data['age_at_admission']
data['recorded_year'] = data['admit_year']
data['bmi_calc'] = data['bmi']
data['ph'] = data['ph_arterial_min']
data['pco2'] = data['pco2_arterial_max']

<h2> ARDS classifer cohort </h2>

In [ ]:
!run_ards.bat

In [ ]:
ards_classifier_cohort = pd.read_csv('ards_classifier_cohort.csv', index_col=0)
ards_classifier_cohort["hospitalizations_joined_id"] = ards_classifier_cohort["hospitalization_id"].str.replace(
    r"^([^_]+_[^_]+).*", r"\1", regex=True)
print(len(ards_classifier_cohort))

# some duplicate rows remain - drop
ards_classifier_cohort = ards_classifier_cohort.drop_duplicates()
print(len(ards_classifier_cohort))

# Some duplicate hospitalizations_joined_id > only difference is cardiac_arrest_primary_dx column
# Remove duplicate cardiac_arres_primary_dx
ards_classifier_cohort = ards_classifier_cohort.sort_values('cardiac_arrest_primary_dx', ascending = False)\
    .drop_duplicates('hospitalizations_joined_id')
print(len(ards_classifier_cohort))

# Merge onto data
print(len(data))
data = data.merge(ards_classifier_cohort[['hospitalizations_joined_id', 'cohort_eligible']], on='hospitalizations_joined_id', how='inner')
data['ards_eligible'] = data['cohort_eligible']
data

<h2> Save final output </h2>

<h3> Table 1 </h3>

In [ ]:
data

<h4> Definition to create table 1 </h4>

In [ ]:
def create_table1(df, continuous_vars, categorical_vars, var_order=None):
    """
    Create a Table 1 summary statistics table

    Parameters
    ----------
    df : pandas DataFrame
        The patient data
    continuous_vars : dict
        {column_name: display_label} for continuous vars
        e.g., {'age': 'Age, median [Q1,Q3]'}
    categorical_vars : dict
        {column_name: display_label} for categorical vars
        e.g., {'race': 'Race, n (%)'}
    var_order : list | None
        Exact row order; use 'n' for the total count.
        Example: ['n', 'age', 'bmi_calc', 'race_category', 'sex_category', ...]

    Returns
    -------
    pandas DataFrame with columns ['Variable','Category','Value']
    """

    results = []

    # Total N
    total_n = len(df)

    def _add_n():
        results.append({'Variable': 'n', 'Category': '', 'Value': str(total_n)})

    def _add_cont(var):
        if var in df.columns:
            label = continuous_vars[var]
            median = df[var].median()
            q1 = df[var].quantile(0.25)
            q3 = df[var].quantile(0.75)
            value = f"{median:.1f} [{q1:.1f},{q3:.1f}]"
            results.append({'Variable': label, 'Category': '', 'Value': value})

    def _add_cat(var):
        if var in df.columns:
            label = categorical_vars[var]
            # header row
            results.append({'Variable': label, 'Category': '', 'Value': ''})
            # category rows
            value_counts = df[var].value_counts()
            for category, count in value_counts.items():
                pct = (count / total_n) * 100
                value = f"{count} ({pct:.1f})"
                results.append({'Variable': '', 'Category': str(category), 'Value': value})

    if var_order is not None:
        # Follow the exact order provided
        for key in var_order:
            if key == 'n':
                _add_n()
            elif key in continuous_vars:
                _add_cont(key)
            elif key in categorical_vars:
                _add_cat(key)
            # else: silently ignore unknown keys
    else:
        # Original behavior (keep as-is)
        _add_n()
        for var, _label in continuous_vars.items():
            _add_cont(var)
        for var, _label in categorical_vars.items():
            _add_cat(var)

    return pd.DataFrame(results)

<h4> Create table 1 </h4>

In [ ]:
table1_data = data.groupby(['hospitalizations_joined_id']).agg(
    age=('age', 'first'),
    bmi_calc=('bmi_calc', 'median'),
    elix_vw=('elix_vw', 'median'),
    sf_ratio=('sf_ratio', 'median'),
    pco2=('pco2', 'median'),
    ph=('ph', 'median'),
    laps2=('laps2', 'median'),
    vent_episode_duration_hours=('vent_episode_duration_hours', 'first'),
    hosp_los_days=('hosp_los_days', 'first'),
    race_category=('race_category', 'first'),
    sex_category=('sex_category', 'first'),
    recorded_year=('recorded_year', 'first'),
    hospital_id=('hospital_id', 'first'),
    mortality_inhospital=('mortality_inhospital', 'first'),
    ards_eligible=('ards_eligible', 'first')
).reset_index()

table1_data

In [ ]:
# Define variables for Table 1
continuous_vars = {
    'age': 'Age, median [Q1,Q3]',
    'bmi_calc': 'BMI, median [Q1,Q3]',
    'elix_vw': 'Elixhauser, median [Q1,Q3]',
    'sf_ratio': 'SF Ratio, median [Q1,Q3]',
    'pco2': 'pCO2, median [Q1,Q3]',
    'ph': 'pH, median [Q1,Q3]',
    'laps2': 'Laps2, median [Q1,Q3]',
    'vent_episode_duration_hours': 'Vent episode duration (hr), median [Q1,Q3]',
    'hosp_los_days': 'Hospitalization length (days), median [Q1,Q3]'
}

categorical_vars = {
    'race_category': 'Race, n (%)',
    'sex_category': 'Sex, n (%)',
    'recorded_year': 'Year, n (%)',
    'hospital_id': 'Hospital, n (%)',
    'mortality_inhospital': 'In-hospitalization mortality, n (%)'
}

var_order = ['n',
             'age', 'race_category', 'sex_category',
             'bmi_calc', 'elix_vw', 'laps2',
             'sf_ratio', 'pco2', 'ph', 'vent_episode_duration_hours',
             'hosp_los_days', 'mortality_inhospital', 'hospital_id', 'recorded_year'
]

# Generate Table 1 (overall only)
table1 = create_table1(table1_data, continuous_vars, categorical_vars, var_order)
table1.to_excel('table1.xlsx', index=False)

# Generate Table 1 stratified by ARDS cohort eligibility
# create_table1 returns: Variable, Category, Value
table1_stratified = table1.rename(columns={'Value': 'Overall'}).copy()

for cohort_value in sorted(table1_data['ards_eligible'].dropna().unique()):
    cohort_df = table1_data[table1_data['ards_eligible'] == cohort_value]
    if cohort_df.empty:
        continue

    if cohort_value in (1, True):
        cohort_label = 'ARDS cohort'
    elif cohort_value in (0, False):
        cohort_label = 'Non-ARDS cohort'
    else:
        cohort_label = f'ards_eligible={cohort_value}'

    cohort_table = create_table1(cohort_df, continuous_vars, categorical_vars, var_order)
    cohort_table = cohort_table.rename(columns={'Value': cohort_label})
    table1_stratified = table1_stratified.merge(
        cohort_table[['Variable', 'Category', cohort_label]],
        on=['Variable', 'Category'],
        how='left'
    )

table1_stratified.to_excel('table1_stratified_by_ards_eligible.xlsx', index=False)

table1, table1_stratified


<h1> Provider counts </h1>

In [ ]:
provider_count_df = data.sort_values(by=['hospitalizations_joined_id', 'date'])
provider_count_df = provider_count_df.drop_duplicates(subset='hospitalizations_joined_id')

counts = provider_count_df['prov_npi_shifted'].value_counts()

median_count = counts.median()
q1 = counts.quantile(0.25)
q3 = counts.quantile(0.75)
iqr = q3 - q1

print(f"Median count: {median_count}")
print(f"Q1: {q1}, Q3: {q3}, IQR: {iqr}")

<h1> Vent mode hours </h1>

In [ ]:
# Collapse to hospitalizaiton per row
vent_mode_hours = data.sort_values(by=['hospitalizations_joined_id', 'date'])
vent_mode_hours = vent_mode_hours.drop_duplicates(subset='hospitalizations_joined_id')

cols = [
    'imv_hours', 'nava_hours', 'blow_by_hours', 'simv_hours', 'other_hours',
    'acvc_hours', 'prvc_hours', 'aprv_hours', 'pressure_control_hours',
    'pressure_support_cpap_hours'
]

# 1) Sum each column
sums = vent_mode_hours[cols].sum()

# 2) Build output df with sum + percent of total imv_hours
vent_mode_summary = pd.DataFrame({
    'sum_hours': sums,
    'percent_of_imv_hours': sums / sums['imv_hours'] * 100
})

# optional: cleaner display
vent_mode_summary = vent_mode_summary.reset_index().rename(columns={'index': 'mode'})
vent_mode_summary['percent_of_imv_hours'] = vent_mode_summary['percent_of_imv_hours'].round(2)

vent_mode_summary

<h1> Save final data to parquet </h1>

In [ ]:
data.to_parquet('LPV_final_data.parquet')
print(f'Hospitalization Number: {len(data['hospitalizations_joined_id'].unique())}')
print(f'Patient Number: {len(data['mdm_link_id'].unique())}')
print(f'Provider Number: {len(data['prov_npi_shifted'].unique())}')
print(f'Date Number: {len(data)}')

In [ ]:
ards_data = data[data['ards_eligible'] == 1]
print(f'Hospitalization Number: {len(ards_data['hospitalizations_joined_id'].unique())}')
print(f'Patient Number: {len(ards_data['mdm_link_id'].unique())}')
print(f'Provider Number: {len(ards_data['prov_npi_shifted'].unique())}')
print(f'Date Number: {len(ards_data)}')
ards_data

In [ ]:
code stopper

In [ ]:
test = data.copy()
test

In [ ]:
hourly_combo = con.sql(f"""
    SELECT
        test.hospitalizations_joined_id,
        test.date,
        test.prov_assigned_hour,
        hourly.hospitalizations_joined_id,
        hourly.recorded_date,
        hourly.recorded_hour,
        hourly.vent_episode_hour_seq
    FROM test
    LEFT JOIN '{hourly_resp_path}' as hourly on
        test.hospitalizations_joined_id = hourly.hospitalizations_joined_id
        AND date(test.date) = date(hourly.recorded_date)
        AND test.prov_assigned_hour = hourly.recorded_hour
""").df()
hourly_combo

In [ ]:
hourly_combo = hourly_combo.sort_values(by=['hospitalizations_joined_id', 'date'])
hourly_combo = hourly_combo.drop_duplicates(subset='hospitalizations_joined_id')
hourly_combo

In [ ]:
median = hourly_combo['vent_episode_hour_seq'].median()
q1 = hourly_combo['vent_episode_hour_seq'].quantile(0.25)
q3 = hourly_combo['vent_episode_hour_seq'].quantile(0.75)
iqr = q3 - q1

print(f"Median: {median:.2f}, IQR: {q1:.2f}–{q3:.2f}")

In [ ]:
hourly_combo[hourly_combo['vent_episode_hour_seq'] <= 24]
# 90% within first vent day

In [ ]:
con.sql(f"""
    SELECT
        h.hospitalizations_joined_id,
        h.recorded_date,
        h.recorded_hour,
        h.vent_episode_id,
        h.vent_episode_hour_seq,
        p.prov_npi_shifted,
        p.prov_name_shifted
    FROM '{hourly_resp_path}' as h
    LEFT JOIN '{provider_path}' as p USING (hospitalizations_joined_id, recorded_date, recorded_hour)
    WHERE h.hospitalizations_joined_id = 'fv100061385_1'
        AND h.device_category = 'imv'
    ORDER BY recorded_date, recorded_hour
""").df()

In [ ]:
con.sql(f"""
    SELECT
        h.hospitalizations_joined_id,
        h.recorded_date,
        h.recorded_hour,
        h.vent_episode_id,
        h.vent_episode_hour_seq,
        p.prov_npi_shifted,
        p.prov_name_shifted
    FROM '{hourly_resp_path}' as h
    LEFT JOIN '{provider_path}' as p USING (hospitalizations_joined_id, recorded_date, recorded_hour)
    WHERE h.hospitalizations_joined_id = 'fv100085907_1'
        AND h.device_category = 'imv'
    ORDER BY recorded_date, recorded_hour
""").df()

In [ ]:
con.sql(f"SELECT * FROM '{provider_path}' where hospitalizations_joined_id = 'fv100085907_1' and recorded_date = '2014-01-16'")

In [ ]:
sns.histplot(hourly_combo['vent_episode_hour_seq'], stat='percent', bins=40)

In [ ]:
hourly_combo[hourly_combo['vent_episode_hour_seq'] >600]

In [ ]:
data[data['hospitalizations_joined_id'] == 'fv147229550_1']

In [ ]:
# con.sql(f"SELECT * FROM '{hourly_resp_path}' where hospitalizations_joined_id = 'fv147229550_1' and recorded_date = '2016-08-29'")

In [ ]:
# con.sql(f"SELECT * FROM '{provider_path}' where hospitalizations_joined_id = 'fv147229550_1' and prov_name_shifted = 'chipman, jeffrey g'")

In [ ]:
con.sql(f"SELECT * FROM '{ecmo_path}' where hospitalizations_joined_id = 'fv147229550_1' order by recorded_time").df()

In [ ]:
# hourly_combo[(hourly_combo['vent_episode_hour_seq'] > 5) & (hourly_combo['vent_episode_hour_seq'] <24) &(hourly_combo['recorded_hour'] == 14)]

In [ ]:
ards_test = test[test['ards_eligible']==1]
ards_test

In [ ]:
test = data[data['ards_eligible'] == 1]
test['ltvv_6'].value_counts()

In [ ]:
test = con.sql(f"""
SELECT
    hospitalizations_joined_id,
    recorded_date,
    recorded_hour,
    tidal_volume_set,
    ibw,
    (tidal_volume_set / ibw) as tidal_volume_set_ibw
FROM '{hourly_resp_path}'
WHERE tidal_volume_set IS NOT NULL
    AND device_category = 'imv'
""").df()

test['ltvv_6'] = test['tidal_volume_set_ibw'] <= 6
test['ltvv_8'] = test['tidal_volume_set_ibw'] <= 8

test['ltvv_6'].value_counts()

In [ ]:
con.sql(f"SELECT * FROM '{hourly_resp_path}'")

In [ ]:
test['ltvv_6'].mean() * 100

In [ ]:
test['ltvv_8'].mean() * 100